<a href="https://colab.research.google.com/github/nitin04-stack/CNN-For-CIFAR/blob/main/CNN_FOR_CIFAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
from torchvision.transforms import transforms


In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
traindataset = CIFAR10(root = "/content/drive/MyDrive/Colab Notebooks./data",train = True,download = True,transform = transform)
testdataset = CIFAR10(root = "/content/drive/MyDrive/Colab Notebooks./data",train = False,download = True,transform = transform)

In [6]:
traindataset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: /content/drive/MyDrive/Colab Notebooks./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [7]:
testdataset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: /content/drive/MyDrive/Colab Notebooks./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [8]:
trainloader = DataLoader(traindataset,batch_size = 64,shuffle = True)
testloader = DataLoader(testdataset,batch_size = 64)

In [9]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()
    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size = 3,padding = 1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128,256),
        nn.ReLU(),

        nn.Linear(256,10)
    )
  def forward(self,x):
    x = self.conv_layers(x)
    x = x.view(x.size(0),-1)  #to make x falatten
    x = self.fc_layers(x)

    return x

In [10]:
model = CNN()

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [12]:
# traning the Cnn

In [13]:
epochs = 10
train_loss = []
val_loss = []
best_val_loss = float("inf")

for epoch in range(epochs):
  model.train()
  training_loss = 0.0
  running_val_loss = 0.0

  for images , labels in trainloader:
    optimizer.zero_grad()
    output = model.forward(images)
    loss = criterion(output,labels)
    loss.backward()
    optimizer.step()
    training_loss += loss.item()
    epoch_train_loss = training_loss/len(trainloader)
    train_loss.append(epoch_train_loss)

  model.eval()
  with torch.no_grad():
    for images , labels in testloader:
      output = model.forward(images)
      loss = criterion(output,labels)
      running_val_loss += loss.item()
      epoch_val_loss = running_val_loss/len(testloader)
      val_loss.append(epoch_val_loss)

  print(f"epochs {epoch+1}/{epochs} =>> training_loss is {epoch_train_loss} , validation loss {epoch_val_loss}")
  if epoch_val_loss < best_val_loss:
    best_val_loss = epoch_val_loss
    torch.save(model.state_dict(),"/content/drive/MyDrive/Colab Notebooks/best_model.pt")


epochs 1/10 =>> training_loss is 1.3499345216909637 , validation loss 1.074249973342677
epochs 2/10 =>> training_loss is 0.9126963271661792 , validation loss 0.8877891256551075
epochs 3/10 =>> training_loss is 0.7291648127233891 , validation loss 0.7831894867359452
epochs 4/10 =>> training_loss is 0.6038802793949766 , validation loss 0.7286284617177999
epochs 5/10 =>> training_loss is 0.4964228385244794 , validation loss 0.7312009015660377
epochs 6/10 =>> training_loss is 0.39957848875342733 , validation loss 0.781046733734714
epochs 7/10 =>> training_loss is 0.31944547191529016 , validation loss 0.8332416568022625
epochs 8/10 =>> training_loss is 0.24647010482676193 , validation loss 0.8801005997095898
epochs 9/10 =>> training_loss is 0.18681579576019208 , validation loss 1.009768807584313
epochs 10/10 =>> training_loss is 0.14877356663393929 , validation loss 1.073771913529961


In [14]:
correct_labels = 0
total_labels = 0

with torch.no_grad():
  for images,labels in testloader:
    output = model.forward(images)
    _,predicted = torch.max(output,1)

    correct_labels += (predicted == labels).sum().item()
    total_labels += labels.size(0)
print(f"accuracy_score {correct_labels / total_labels *100}")

accuracy_score 75.56
